# 结构恢复：交互式实验

本 notebook 把中文镜像站中的 [Cookbook](/cookbooks/autoformat/) 改写成可以逐格运行、修改输入并观察结果的最小实验。
判断换行边界并从纯文本重建段落。

运行方式与 `../03_架构模式/01_架构模式.ipynb` 一致：有有效的 `TYPESAFE_API_KEY` 时调用真实的
TypeSafe API；没有 Key 或返回 401 时使用内置的离线示例答案。后续代码不区分两种模式，便于先学习
控制流，再切换到真实模型观察概率和置信度。

> 学习提示：先顺序运行全部单元格，再回到“定义 state”或“定义问题”的单元格修改内容，重新运行后面的单元格。
> API Key 只从环境变量读取，不能写进 notebook。


## 0. 准备

### 0.1 安装依赖

In [1]:
%pip install -q -U typesafe-sdk

Note: you may need to restart the kernel to use updated packages.


### 0.2 创建客户端

In [2]:
import os
import statistics
import time
from pprint import pprint

from typesafe_sdk import (
    Choice,
    Score,
    Noul,
    TypeSafeClient,
    TypeSafeAuthenticationError,
)

API_KEY = os.environ.get("TYPESAFE_API_KEY", "")
client = TypeSafeClient(api_key=API_KEY, model="jev-latest") if API_KEY else None
print("客户端已创建：模型=jev-latest，Key=", "已配置" if API_KEY else "未配置（将使用离线示例）")


客户端已创建：模型=jev-latest，Key= 已配置


### 0.3 离线响应与统一调用入口

In [3]:
class _FakeAnswer:
    def __init__(self, type_, **values):
        self.type = type_
        for key, value in values.items():
            setattr(self, key, value)


class _FakeResponse:
    def __init__(self, answers):
        self.answers = answers
        self.nouls = {k: v for k, v in answers.items() if v.type == "noul"}
        self.choices = {k: v for k, v in answers.items() if v.type == "choice"}
        self.scores = {k: v for k, v in answers.items() if v.type == "score"}
        self.model = "jev-latest（离线示例）"
        self.usage = _FakeAnswer("usage", input_tokens=0, output_tokens=0)


class TS:
    offline = False
    _warned = False

    @classmethod
    def call(cls, state, questions, offline_answers):
        if client is None:
            cls.offline = True
            if not cls._warned:
                cls._warned = True
                print("⚠️ 未设置有效 TYPESAFE_API_KEY，以下输出使用内置离线示例。")
            return _FakeResponse(offline_answers)
        try:
            return client.system_one(state, questions)
        except TypeSafeAuthenticationError:
            cls.offline = True
            if not cls._warned:
                cls._warned = True
                print("⚠️ 未设置有效 TYPESAFE_API_KEY，以下输出使用内置离线示例。")
            return _FakeResponse(offline_answers)


def answer_line(name, answer):
    if answer.type == "noul":
        return f"{name}: noul={answer.noul:.2f}"
    if answer.type == "choice":
        return f"{name}: choice={answer.choice} confidence={answer.confidence:.2f}"
    return f"{name}: score={answer.score:.2f} confidence={answer.confidence:.2f}"


print("模式：", "离线示例" if TS.offline else "真实 API（首次调用后确定）")


模式： 真实 API（首次调用后确定）


### 0.4 连通性测试

In [4]:
if client is None:
    TS.offline = True
    print("⚠️ API Key 未设置，后续单元格使用离线示例。")
else:
    try:
        ping = client.system_one("你好", {"is_greeting": Noul(instructions="这段文字是在打招呼吗？")})
        print("✅ API 连通正常，后续单元格会使用真实结果。")
    except TypeSafeAuthenticationError:
        TS.offline = True
        print("⚠️ API Key 无效，后续单元格使用离线示例。")


✅ API 连通正常，后续单元格会使用真实结果。


## 1. 结构恢复

判断相邻文本行之间的换行是否切断了同一句话，用 Noul 得到连接概率，再按阈值重建段落。


### 1.1 定义被错误换行的文本

In [5]:
LINES = [
    "TypeSafe 返回结构化答案，",
    "代码可以直接消费这些答案。",
    "每个问题都应当足够窄，",
    "让模型在一秒内完成判断。",
    "这是一个新的段落。",
]
print("待恢复文本已定义：行数=", len(LINES))


待恢复文本已定义：行数= 5


### 1.2 判断每个相邻行是否属于同一句

In [6]:
JOIN_PROBABILITIES = [.91, .87, .78, .12]
joins = []
for index in range(len(LINES) - 1):
    pair = {"left": LINES[index], "right": LINES[index + 1]}
    response = TS.call(pair,
                       {"same_sentence": Noul(instructions="换行是否把同一个句子切开了？")},
                       {"same_sentence": _FakeAnswer("noul", noul=JOIN_PROBABILITIES[index])})
    probability = response.nouls["same_sentence"].noul
    joins.append(probability)
    print(f"{probability:.2f}  {LINES[index]} + {LINES[index + 1]}")


0.70  TypeSafe 返回结构化答案， + 代码可以直接消费这些答案。


0.48  代码可以直接消费这些答案。 + 每个问题都应当足够窄，


0.79  每个问题都应当足够窄， + 让模型在一秒内完成判断。


0.25  让模型在一秒内完成判断。 + 这是一个新的段落。


### 1.3 根据概率重建段落

In [7]:
paragraphs = [LINES[0]]
for index, probability in enumerate(joins):
    if probability >= 0.50:
        paragraphs[-1] += LINES[index + 1]
    else:
        paragraphs.append(LINES[index + 1])

print("\n\n".join(paragraphs))


TypeSafe 返回结构化答案，代码可以直接消费这些答案。

每个问题都应当足够窄，让模型在一秒内完成判断。

这是一个新的段落。


观察：模型只判断相邻两行是否属于同一句，拼接和分段仍由代码控制。调高阈值会产生更多段落，调低阈值会更激进地合并。

## 知识补充
- **判断不生成**：让模型判"这里该不该换段"（noul），重建文本由代码拼接——原文一字不丢。这是 Jev 与生成式模型的分界线：凡是要保真的处理，都用判断+代码，不要摘要。
- **生产实例**（jev-cookbook 04）：fast-jev-compaction 插件用同样模式给 Claude Code 做上下文压缩——逐条工具调用打分、过期的丢、保留的保持原文，替代有损摘要。
- **批量优势**：这类任务往往百万行级，Jev 按输入计费（$0.042/M token、输出免费），成本约为大模型调用的百分之一量级。

## 小结

这本 notebook 的边界很清楚：TypeSafe 只负责受限、可编程的判断；排序、阈值、分组、重建文本和
函数分派都由 Python 代码完成。修改输入或问题后重新运行，就能观察“模型答案 → 确定性代码”的变化。
